# 第7回　データの整理と比較：クロス集計
## ―― 全体で見ると差があるのに、グループごとに見ると差が消えるのはなぜか

情報活用　／　北星学園大学　2026年度後期

今日つくる表は **クロス集計表（分割表）**。2つの項目を同時に数えた表である。
これはこの授業の中核であり、あなたの報告書の中心になる表でもある。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語が豆腐（□）にならないようにする

def _build_from_source(keep_missing_code=False):
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    out = out.sort_values("学名").reset_index(drop=True)
    return out if keep_missing_code else out.replace(-999, np.nan)

try:
    df = pd.read_csv("https://aonoa68.github.io/joho-katsuyo/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 0. なぜピボットテーブルを使わないのか

Excelには「ピボットテーブル」という、同じ表を作る便利な機能がある。**それは使わない。**
作る表は同じである。使わないのは機能のほうだ。理由は1つ。

> **何をどう数えたかが記録されないため、あとから再現できず、他人も検証できない。**

ピボットテーブルは、マウスで列をドラッグして作る。半年後に「この表、どうやって作ったんですか」と聞かれても、**操作の履歴はどこにも残っていない**。
コードなら、どの列を、どの条件で、どう数えたかが全部残る。もう一度実行すれば同じ表が出る。

**結果だけが出てきて過程が残らない道具は、検証に使えない。**
これは、AIの出力に対してこの授業がとる態度と、まったく同じ構造である。

> **このあと `pivot_table` という名前の関数が出てくるが、Excelのピボットテーブルとは別物である。**
> 名前は同じでも、こちらは**コードとして残る**ので、何をどう数えたかがあとから読める。
> 使わないのは「**マウス操作で作って履歴が残らない**」ほうであって、**表の作り方そのものではない。**

> **この考え方は、この授業が道具を選ぶときの基準そのものである。**
> → 授業サイト「過程が残る道具を使う」 https://aonoa68.github.io/joho-katsuyo/process/


---
## 1. 1973年、カリフォルニア大学バークレー校

大学院の合格率に男女差がある、と大問題になった事件がある。まず全体の数字を見る。

In [ ]:
# 1973年 バークレー校 大学院 出願者の多い6学部の実データ
# 出典: Bickel, Hammel & O'Connell (1975) Science, 187(4175), 398-404.
berkeley = pd.DataFrame({
    "学部": ["A","A","B","B","C","C","D","D","E","E","F","F"],
    "性別": ["男","女"]*6,
    "出願": [825,108, 560,25, 325,593, 417,375, 191,393, 373,341],
    "合格": [512, 89, 353,17, 120,202, 138,131,  53, 94,  22, 24],
})
berkeley["不合格"] = berkeley["出願"] - berkeley["合格"]

total = berkeley.groupby("性別")[["出願","合格"]].sum()
total["合格率"] = (total["合格"] / total["出願"] * 100).round(1)
total

**男性 44.5%、女性 30.4%。** 14ポイントもの差がある。

（よく引用される「男44%・女35%」は大学全体の数字。ここで見ているのは出願者の多い上位6学部で、差はさらに大きい。）

これだけ見れば「女性が不利に扱われている」と読める。ところが、**学部ごとに分けて**数え直すと、話が変わる。

In [ ]:
by_dept = berkeley.pivot_table(index="学部", columns="性別", values=["出願","合格"], aggfunc="sum")
rate = (by_dept["合格"] / by_dept["出願"] * 100).round(1)
rate["女が高い?"] = np.where(rate["女"] > rate["男"], "← 女性のほうが高い", "")
rate

**6学部中4学部で、女性のほうが合格率が高い。** 残り2つも差はわずかである。

全体では女性が14ポイント低いのに、どの学部を見ても女性が不利ではない。なぜこうなるのか。

In [ ]:
apply = by_dept["出願"].copy()
apply["学部全体の合格率"] = (by_dept["合格"].sum(axis=1) / by_dept["出願"].sum(axis=1) * 100).round(1)
apply["女性の出願割合"] = (apply["女"] / (apply["男"] + apply["女"]) * 100).round(1)
apply.sort_values("学部全体の合格率", ascending=False)

答えが出た。**合格率の高い学部A・Bには男性が集中し（A 88%・B 96%が男性）、合格率の低い学部C・Eに女性が集中している**（どちらも女性が65〜67%）。最難関の学部Fは、男女ほぼ半々である。

学部Aの合格率は64%、学部Fは6%。**そもそも入りやすさが10倍違う。**
女性は難関の学部に多く出願していたので、全体をまぜると合格率が下がって見えた。

> **全体の数字は「差別」、分けると「逆」。どちらが本当かは、分けて初めてわかる。**

これを **シンプソンのパラドックス** という。

### なぜ起こるのか（ひとことで）

```
グループごとに「件数の偏り」や「別の要因」があると、
全体の平均は、その偏りに引っぱられて本当の傾向を隠す（時に逆転させる）
```

平均や全体は、**たった1個にまとめた数字**である。まとめる過程で、グループの情報が消える。

### 数字を見たときの合言葉

1. これ、**何かで分けたら違う話にならないか**（性別・学年・地域・時期・グループ）
2. どこかの**グループが極端に多く／少なく**混じっていないか

---
## 2. クロス集計表を自分で作る

使う関数は `pd.crosstab`。**行に置く項目**と**列に置く項目**を渡すだけ。

霊長類データで「**科によって食べ物が違うのか**」を見る。栄養段階は 1＝草食、2＝雑食 で記録されている。

In [ ]:
d = df.copy()
d["食性"] = d["栄養段階"].map({1: "草食", 2: "雑食"})

# 種数の多い科だけに絞る（少なすぎる科は比較にならない）
big = d["科"].value_counts()
big = big[big >= 15].index
d = d[d["科"].isin(big)]

pd.crosstab(d["科"], d["食性"])

件数の表ができた。ただし **件数のままでは比べられない。**
科ごとに種数が違うので、「オナガザル科は草食が多い」と言っても、単にオナガザル科の種数が多いだけかもしれない。

**割合に直す。** `normalize="index"` で、行ごとに合計100%にする。

In [ ]:
tab = pd.crosstab(d["科"], d["食性"], normalize="index") * 100
tab.round(1)

**オマキザル科とコビトキツネザル科は、記録のある種がすべて雑食。**一方クモザル科は7割が草食（葉や果実を食べる）。科によってはっきり違う。

In [ ]:
# 合計も一緒に出す（報告書にはこの形で載せる）
pd.crosstab(d["科"], d["食性"], margins=True, margins_name="合計")

> **どちらの方向で割ったかを必ず書く。**
> `normalize="index"` は行方向（科ごとに100%）、`normalize="columns"` は列方向（草食の中で100%）。
> 同じ表から、まったく違う主張が作れてしまう。

> **そして、この表の分母は「栄養段階が記録されている種」だけである。**
> 第6回で見たとおり、栄養段階は376種中178種（47%）しか埋まっていない。
> **記録のない種が、どちらかに偏っていない保証はない。**

---
## 3. 層別を1つ加える ―― 結論は変わるか

> **層別とは、データを3つめの項目でグループに分けて、グループごとに同じ表を作ること。**
> 全体で1枚だった表が、グループの数だけになる。
> **さっきのバークレーが、まさにこれである。**全体で1枚 → 学部ごとに6枚にしたら、結論が逆になった。

バークレーでやったのと同じことをする。**3つめの項目で分けてみる。**



In [ ]:
# 体の大きさで2つに分ける（中央値で区切る）
e = df.dropna(subset=["体重g", "集団サイズ"]).copy()
e["体格"] = np.where(e["体重g"] >= e["体重g"].median(), "大きい", "小さい")

print("【全体】体格別の平均集団サイズ")
print(e.groupby("体格")["集団サイズ"].agg(["count", "mean"]).round(1))

In [ ]:
# 次に、科で分けて見る
# ★ 平均だけでなく「何種から出した平均か」を必ず一緒に出す（第6回でやったこと）
split = e.pivot_table(index="科", columns="体格", values="集団サイズ",
                      aggfunc=["mean", "count"])
split.columns = [f"{'平均' if a=='mean' else '種数'}_{b}" for a, b in split.columns]
split["科の種数"] = e.groupby("科").size()
split[split["科の種数"] >= 10].round(1)


### この表の読み方 ―― 平均だけを見ない

**オナガザル科の「小さい」は、91.2。**全体では「大きい 17.9／小さい 6.6」だったのに、この科だけ逆転している
――ように**見える**。

**種数の列を見てほしい。1種である。**
タラポアン（体重1.2kg・集団サイズ91.2）という1種の値が、そのまま「平均」として表に出ている。
69種から出した 20.5 と、1種の 91.2 を、同じ列に並べた表だった。

> **n=1の平均は、平均ではない。**その1個の値である。
> 第6回で「**どの数字が何件から出ているかを、いつも一緒に示す**」と言ったのは、このためだ。
> 件数を出さずにこの表を見た人は、**ありもしない逆転を見つけてしまう。**

### `NaN` は壊れているのではない

オマキザル科とガラゴ科の「大きい」、クモザル科とテナガザル科の「小さい」が `NaN` になっている。
**その組み合わせに当てはまる種が1つもない**、という意味である。
オマキザル科には、体重3.1kg（全体の中央値）を超える種がいない。**データがないことも、1つの結果である。**

### 結論：**この層別では、比べられない**

もう一度、種数の列だけを見る。

- オナガザル科：小さいが **1種**
- キツネザル科：大きいが **1種**
- サキ科：大きいが **2種**
- オマキザル科・ガラゴ科：大きいが **0種** ／ クモザル科・テナガザル科：小さいが **0種**

**どの科も、片側が足りない。**だから「逆転した」とも「逆転しなかった」とも言えない。
**出せるのは「この分け方では比べられない」という結論である。**それは失敗ではない。確かめた結果である。

> **層別は、分ければ分けるほど1マスが小さくなる。**
> あなたの調査も同じだ。**30人の回答を2×2に分ければ、1マスは7〜8人**になる。
> そこで出た「平均」は、第6回で見たとおり**毎回ちがう数字**になる。
>
> だから報告書には、**表のマスごとの件数を必ず載せる。**
> 第8回（順位は差の大きさを消す）、第13回（この件数で、どこまで言えるか）に、そのまま続く話である。

**確かめずに「差がある」と書くことだけが、してはいけないことである。**


---
## 4. 自分の調査データでやる

ここから先は、**自分の調査データ**でやる。することは5つ。

1. **CSVを読み込む**（すぐ下のセル）
2. 数値のはずの列が文字列になっていたら直す（**必要になった人だけ**）
3. **件数**のクロス集計表を作る
4. **割合**の表にする
5. **層別を1つ加える**（3つめの項目で分ける）

そのあと「5. 卒業」で、同じ表を2通りの書き方で作って突き合わせる。

> **2. は、自分のデータで必要になった人だけがやる。**
> 読み込んで列名が出たら、そのまま 3.（クロス集計のセル）まで進んでよい。
> エラーが出たときに、2. に戻ってくればよい。


In [ ]:
# ★ 使う行は、先頭の # を消してから ▶ を押す（# が付いた行は実行されません）
# ここから先は「自分の調査データ」でやる。
# Google Forms の回答 → スプレッドシート → ファイル → ダウンロード → CSV で書き出したものを使う。
#
# 左のフォルダアイコン（📁）にCSVをドラッグしてから、ファイル名を書き換えて実行する。
# ※ 数字を手で打ち直さないこと。転記した瞬間に、それは元データではなくなる。

# mydf = pd.read_csv("自分のファイル名.csv")
# mydf.head()

### その前に ―― 「数値のはずの列」が文字列になっていたら

第6回で、平均を出そうとしてこうなった人がいる。

```
TypeError: Could not convert string '...' to numeric
```

**読み込みは成功している。**止まっているのはその先で、**列の中身が文字列**だということ。
第6回にやった「数値の列に文字が混ざっている」が、自分のデータで起きた形である。
**1つ混ざるだけで、列全体が文字列になる。**

このクラスで実際に起きたのは、次の2つだった。

| 起きたこと | 列に入っていた値 |
|---|---|
| 自由入力の金額欄に、文章が書き込まれていた | `単位がないのでどう書いたらいいか不明` |
| 5段階の選択肢が、ラベルごと入っていた | `1：全く当てはまらない` |

**順番は、① 犯人を探す → ② 直す。**いきなり直さないこと。
どの回答が原因かを見ないまま消すと、**消してはいけないものまで消える。**

今日は層別（中央値で2つに分ける）で数値の列を使う。**そこに入る前に直しておく。**


In [ ]:
# ★ 使う行は、先頭の # を消してから ▶ を押す（# が付いた行は実行されません）
# ① 犯人を探す ―― 数値にできない値が、どこに入っているかを見る

# col = "ここに列名"
# bad = pd.to_numeric(mydf[col], errors="coerce").isna() & mydf[col].notna()
# print(f"数値にできない回答: {bad.sum()} 件 / 回答のあった {mydf[col].notna().sum()} 件中")
# print(mydf.loc[bad, col].to_string())      # ← 実物を見る。ここを飛ばさない


In [ ]:
# ★ 使う行は、先頭の # を消してから ▶ を押す
# ②-A ラベル付きの選択肢から、先頭の数字だけを取り出す
#      例：「1：全く当てはまらない」→ 1 ／「5：とても当てはまる」→ 5

# col = "ここに列名"
# mydf[col] = mydf[col].astype(str).str.extract(r"^\s*(\d+)").astype(float)
# print(mydf[col].describe())


In [ ]:
# ★ 使う行は、先頭の # を消してから ▶ を押す
# ②-B 単位や記号が付いた数字を、数値にする
#      例：「7.2kg」→ 7.2 ／「1,200円」→ 1200 ／「60分」→ 60

# col = "ここに列名"
# mydf[col] = pd.to_numeric(
#     mydf[col].astype(str).str.replace(r"[^0-9.]", "", regex=True),
#     errors="coerce")
# print(f"数値にできなかった（欠測にした）: {mydf[col].isna().sum()} 件")
# print(mydf[col].describe())


**直したら、何をしたかを書き残す。**

- どの列の、**何件が**、どんな値だったか
- それをどう扱ったか（数字だけ取り出した／欠測にした／そのまま残した）
- **なぜそうしてよいと判断したか**

`errors="coerce"` は、**数値にできなかったものを、黙って欠測（NaN）に変える。**
便利だが、**何件消えたかを見ないまま使ってはいけない。**上のセルで件数を出しているのは、そのためである。

> 本当に直すべきなのは、**次に調査票を作るとき**である。
> 「1週間のアルバイト代はいくらですか」ではなく
> 「**1週間のアルバイト代は何円ですか（半角数字のみ／例：8000）**」と書いてあれば、この作業は要らなかった。


In [ ]:
# ★ 使う行は、先頭の # を消してから ▶ を押す（# が付いた行は実行されません）
# クロス集計（列名を自分のものに書き換える）
# pd.crosstab(mydf["行にする項目"], mydf["列にする項目"], margins=True)

# 割合にする
# (pd.crosstab(mydf["行にする項目"], mydf["列にする項目"], normalize="index") * 100).round(1)

In [ ]:
# ★ 使う行は、先頭の # を消してから ▶ を押す
# 層別：3つめの項目で分けて、同じ表をグループごとに作る
# 「全体で見えた差は、分けたあとも残るか」を確かめる

# row, col, by = "行にする項目", "列にする項目", "分けるのに使う項目"
# for name, g in mydf.groupby(by):
#     print(f"=== {by} = {name}（{len(g)}件）===")
#     print(pd.crosstab(g[row], g[col], margins=True))                          # 件数
#     print((pd.crosstab(g[row], g[col], normalize="index") * 100).round(1))    # 割合
#     print()

# ※ 分けるほど1マスが小さくなる。件数（上の表）を必ず一緒に見ること。
#    1マスが1人でも「100%」と出てしまう。


---
## 5. 卒業 ―― 同じ表が、別の道でも出るか

同じクロス集計表を、**2通りの書き方**で作って一致するか確かめる。

1. `pd.crosstab(...)`
2. `df.groupby([行, 列]).size().unstack(fill_value=0)`

**同じ数字が出れば、少なくとも書き間違いではない。**ずれたら、原因はたいてい次のどれかである。

- **空欄の数え方。**どちらも空欄の行は自動で除く。`fill_value=0` を付けると、「0件だった」と「そもそも集計から外れた」が**見分けられなくなる**
- **割る方向。**`normalize="index"`（行ごと100%）か `normalize="columns"`（列ごと100%）か
- **列の取り違え**

**何件から作った表かを、必ず書く。**

> 道具を変えても同じ数字が出るか確かめる、というのがここでやっていることである。
> 便利な機能を1つ覚えて終わりにしないのは、**確かめる手段を自分の手元に残しておく**ためである。

そして、**この表を作らずに同じことが言えるか**も考えてみる。言えるなら、その表はいらない。言えないなら、その表があなたの主張の根拠である。


In [ ]:
# ★ 使う行は、先頭の # を消してから ▶ を押す
# 同じ表を、別の書き方でもう一度作って一致するか確かめる

# row, col = "行にする項目", "列にする項目"
# a = pd.crosstab(mydf[row], mydf[col])
# b = mydf.groupby([row, col]).size().unstack(fill_value=0)
# print(a)
# print(b)
# print("一致:", a.equals(b))      # False なら、上の2つを目で見比べる


---
## AIとのやり取りを貼る（1点）

**今日AIに聞いたやり取りを、下のセルに貼ってください。1回ぶんで構いません。**

- **内容では採点しません。**貼ってあることだけを見ます。
- うまくいかなかったやり取り、見当違いの答えが返ってきたやり取りでも構いません。**むしろそちらのほうが役に立ちます。**
- AIを使わなかった回は、「使わなかった」と書いてください。それで1点です。

> 貼るという行為そのものが、**AIの答えを読んで自分の状況に当てはめる**作業になります。
> 流し読みで終わらせないための仕掛けでもあります。


In [ ]:
# ★ AIとのやり取りを、下の """ と """ のあいだに貼る。
#    ▶ を押す必要はありません。貼るだけでかまいません。

ai_log = """
【聞いたこと】


【返ってきた答え】


"""


---
## 課題7（8点）

**うち1点は、AIとのやり取りの貼り付け**（ひとつ上のセル）。残り7点が以下。

**このノートブック（コードと出力）** ＋ **解釈3文**。

- [ ] 自分のデータのクロス集計表（件数と割合の両方）
- [ ] 層別を1つ加えた表
- [ ] 解釈3文。**うち1文は「層別したら何が変わったか」**（変わらなかったなら、そう書く）
- [ ] **2通りの書き方で同じ表になったか。**ずれた場合は原因
- [ ] **AIとのやり取りの貼り付け（1点）。**内容では採点しない。貼ってあることだけを見る

提出期限：次回授業の開始まで（遅れた場合は50%）

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
